# 13 - Streamlit Documentation Chatbot

This notebook follows the instructor's `main.py` structure.

```text
Streamlit chat input
→ backend.core.run_llm()
→ Gemini agentic RAG backend
→ answer and Document context
→ Streamlit answer and source links
```

No Gemini-specific changes are required in the frontend because Gemini is handled inside `backend/core.py`.

The code below is intended to become `main.py` when the application is run with Streamlit.

## 1. Optional installation

In [ ]:
# !pip install -U streamlit

## 2. Imports

The SSL and `.env` setup is kept at the start, following the course notebook convention.

In [1]:
import truststore
truststore.inject_into_ssl()

from pathlib import Path
ENV_PATH = Path("../langchain-course/.env").resolve()

from dotenv import load_dotenv
load_dotenv(dotenv_path=ENV_PATH)

from typing import Any, Dict, List
import streamlit as st

from backend.core import run_llm

ModuleNotFoundError: No module named 'backend'

## 3. Format source URLs

The backend returns LangChain `Document` objects in `context`. This helper extracts each document's `metadata["source"]` value.

In [ ]:
def _format_sources(
    context_docs: List[Any],
) -> List[str]:
    return [
        str(meta.get("source") or "Unknown")
        for doc in (context_docs or [])
        if (
            meta := (
                getattr(doc, "metadata", None)
                or {}
            )
        ) is not None
    ]


## 4. Configure the Streamlit page

In [ ]:
st.set_page_config(
    page_title="LangChain Documentation Helper",
    layout="centered",
)

st.title("LangChain Documentation Helper")


## 5. Add the sidebar and clear-chat button

In [ ]:
with st.sidebar:
    st.subheader("Session")

    if st.button(
        "Clear chat",
        use_container_width=True,
    ):
        st.session_state.pop(
            "messages",
            None,
        )
        st.rerun()


## 6. Initialize chat history

In [ ]:
if "messages" not in st.session_state:
    st.session_state.messages = [
        {
            "role": "assistant",
            "content": (
                "Ask me anything about LangChain docs. "
                "I'll retrieve relevant context and cite sources."
            ),
            "sources": [],
        }
    ]


## 7. Display existing chat messages

In [ ]:
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

        if msg.get("sources"):
            with st.expander("Sources"):
                for source in msg["sources"]:
                    st.markdown(f"- {source}")


## 8. Accept a user question and call the backend

In [ ]:
prompt = st.chat_input(
    "Ask a question about LangChain..."
)

if prompt:
    st.session_state.messages.append({
        "role": "user",
        "content": prompt,
        "sources": [],
    })

    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        try:
            with st.spinner(
                "Retrieving docs and generating answer..."
            ):
                result: Dict[str, Any] = run_llm(
                    prompt
                )

            answer = (
                str(result.get("answer", "")).strip()
                or "(No answer returned.)"
            )

            sources = _format_sources(
                result.get("context", [])
            )

            st.markdown(answer)

            if sources:
                with st.expander("Sources"):
                    for source in sources:
                        st.markdown(f"- {source}")

            st.session_state.messages.append({
                "role": "assistant",
                "content": answer,
                "sources": sources,
            })

        except Exception as error:
            st.error(
                "Failed to generate a response."
            )
            st.exception(error)


## Running the application

After the backend notebook is converted to `backend/core.py` and this frontend code is saved as `main.py`, run:

```bash
streamlit run main.py
```

The browser application will normally open at `http://localhost:8501`.